# Tutorial 1.2, Part 1: Time-series data quality and imputation

## 1. Overview and learning outcomes

This tutorial accompanies Lecture 1.2 of **BENV0169 Data Analytics for Sustainable Buildings**. We use hourly aggregate electricity demand to inspect missing data and compare ways to fill artificial gaps.

By the end of this part, you should be able to:

- inspect a time index and distinguish absent timestamps from stored `NaN` values;
- compare imputation methods using observations that we hide artificially;
- evaluate reconstructed values only where the true observations were hidden;
- use plots to explain where an imputation method works well or poorly.

### Indicative schedule

| Activity | Time |
|---|---:|
| Setup and time-index checks | 10 minutes |
| Imputation experiment and plots | 25 minutes |
| Interpretation and optional extension | 10 minutes |

Forecasting is a separate tutorial: [Part 2: Introduction to day-ahead load forecasting](tutorial_1_2_forecast_intro.ipynb).

## 2. Setup and data loading

The data come from the load-forecasting track of GEFCom2012. Columns `Z1` to `Z20` contain demand for twenty zones. `Z21` is their total demand. Demand is measured in MW.

**Reference**

Hong, T., Pinson, P. and Fan, S. (2014) ‘Global Energy Forecasting Competition 2012’, *International Journal of Forecasting*, 30(2), pp. 357–363. [https://doi.org/10.1016/j.ijforecast.2013.07.001](https://doi.org/10.1016/j.ijforecast.2013.07.001)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["figure.constrained_layout.use"] = True
plt.rcParams["figure.dpi"] = 110
plt.rcParams["figure.figsize"] = (10, 4)


In [ ]:
DATA_PATH = Path("data") / "tutorial_1_2" / "load_data_full.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Launch Jupyter from the repository root."
    )

load_data = pd.read_csv(DATA_PATH, index_col=0, parse_dates=[0])
load_data.index.name = "timestamp"
load_data = load_data.sort_index()

if not isinstance(load_data.index, pd.DatetimeIndex):
    raise TypeError("The first CSV column must parse as a DatetimeIndex.")
if "Z21" not in load_data.columns:
    raise KeyError("Expected the aggregate load column 'Z21'.")

print(f"Loaded {len(load_data):,} rows and {load_data.shape[1]} demand columns.")
load_data.head(3)


## 3. Time-index quality

### Worked example: inspect the time index

Before analysing a time series, check its dates, sampling interval, duplicate timestamps and missing data. The supplied code reports:

- the start and end timestamps;
- the usual time between observations;
- duplicate timestamps;
- timestamps missing from the expected hourly sequence;
- stored `NaN` values.

Run the cells and inspect the summary and missing-block table. Identify absent timestamps **before** reindexing the data onto the complete hourly time grid. You do not need to complete any code in this section.

In [ ]:
def summarise_time_index(data, expected_frequency="h"):
    '''Return key time-index and explicit-missingness checks.'''
    if not isinstance(data.index, pd.DatetimeIndex):
        raise TypeError("data must have a DatetimeIndex.")
    if data.empty:
        raise ValueError("data must contain at least one row.")

    time_differences = data.index.to_series().diff().dropna()
    nominal_interval = time_differences.mode().iloc[0]
    expected_index = pd.date_range(
        start=data.index.min(),
        end=data.index.max(),
        freq=expected_frequency,
    )
    missing_timestamps = expected_index.difference(data.index)

    return {
        "start": data.index.min(),
        "end": data.index.max(),
        "nominal_interval": nominal_interval,
        "duplicate_timestamps": int(data.index.duplicated().sum()),
        "missing_timestamps": missing_timestamps,
        "rows_with_nan": int(data.isna().any(axis=1).sum()),
        "nan_cells": int(data.isna().sum().sum()),
    }


In [ ]:
time_quality = summarise_time_index(load_data)

print("Data-quality summary")
print(f"  Coverage: {time_quality['start']} to {time_quality['end']}")
print(f"  Nominal interval: {time_quality['nominal_interval']}")
print(f"  Duplicate timestamps: {time_quality['duplicate_timestamps']:,}")
print(f"  Missing timestamps: {len(time_quality['missing_timestamps']):,}")
print(f"  Stored rows containing NaN: {time_quality['rows_with_nan']:,}")
print(f"  Explicit NaN cells: {time_quality['nan_cells']:,}")


In [ ]:
def consecutive_timestamp_blocks(timestamps, frequency="h"):
    '''Summarise a DatetimeIndex as consecutive blocks.'''
    timestamps = pd.DatetimeIndex(timestamps).sort_values()
    if timestamps.empty:
        return pd.DataFrame(columns=["start", "end", "hours"])

    step = pd.to_timedelta(1, unit=frequency)
    block_starts = [timestamps[0]]
    block_ends = []

    for previous, current in zip(timestamps[:-1], timestamps[1:]):
        if current - previous != step:
            block_ends.append(previous)
            block_starts.append(current)
    block_ends.append(timestamps[-1])

    blocks = pd.DataFrame({"start": block_starts, "end": block_ends})
    blocks["hours"] = (
        (blocks["end"] - blocks["start"]) / pd.Timedelta(hours=1) + 1
    ).astype(int)
    return blocks


missing_blocks = consecutive_timestamp_blocks(time_quality["missing_timestamps"])
missing_blocks


In [ ]:
if time_quality["duplicate_timestamps"] != 0:
    raise ValueError("Resolve duplicate timestamps before reindexing.")

full_hourly_index = pd.date_range(
    start=load_data.index.min(),
    end=load_data.index.max(),
    freq="h",
    name="timestamp",
)
hourly_load_data = load_data.reindex(full_hourly_index)

newly_created_missing_rows = hourly_load_data.index.difference(load_data.index)
print(f"Rows after hourly reindexing: {len(hourly_load_data):,}")
print(f"Rows inserted by reindexing: {len(newly_created_missing_rows):,}")


**Interpretation.** An absent timestamp has no row in the original data. We find it by comparing the observed timestamps with the complete hourly sequence. A stored `NaN` already has a row, but its value is missing. Reindexing adds rows for absent timestamps and places `NaN` in them. Count the two types separately before reindexing.

### Seasonal patterns in a complete year

The 2007 data have a complete hourly index. The first plot shows two weeks of `Z21`. The other plots show mean demand by hour of day and hour of week. Use them to identify daily seasonality, $m=24$, and weekly seasonality, $m=168$.

In [ ]:
aggregate_load_2007 = hourly_load_data.loc["2007", "Z21"].copy()
expected_2007_index = pd.date_range(
    "2007-01-01 00:00", "2007-12-31 23:00", freq="h", name="timestamp"
)

if not aggregate_load_2007.index.equals(expected_2007_index):
    raise ValueError("The selected 2007 index is not complete and hourly.")
if aggregate_load_2007.isna().any():
    raise ValueError("The selected 2007 aggregate series contains missing values.")

two_weeks = aggregate_load_2007.loc["2007-10-01":"2007-10-14 23:00"]
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(two_weeks.index, two_weeks, color="tab:blue", linewidth=1.2)
ax.set(
    title="Aggregate hourly demand over two weeks",
    xlabel="Timestamp",
    ylabel="Demand (MW)",
)
ax.grid(alpha=0.25)
plt.show()

mean_by_hour = aggregate_load_2007.groupby(aggregate_load_2007.index.hour).mean()
hour_of_week = aggregate_load_2007.index.dayofweek * 24 + aggregate_load_2007.index.hour
mean_by_hour_of_week = aggregate_load_2007.groupby(hour_of_week).mean()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(mean_by_hour.index, mean_by_hour, marker="o", markersize=3)
axes[0].set(
    title="Mean demand by hour of day",
    xlabel="Hour of day",
    ylabel="Mean demand (MW)",
)
axes[0].set_xticks(range(0, 24, 3))
axes[0].grid(alpha=0.25)

axes[1].plot(mean_by_hour_of_week.index, mean_by_hour_of_week)
axes[1].set(
    title="Mean demand by hour of week",
    xlabel="Hour of week (Monday 00:00 = 0)",
    ylabel="Mean demand (MW)",
)
axes[1].set_xticks(range(0, 169, 24))
axes[1].grid(alpha=0.25)
plt.show()


## 4. Imputation experiment

The long gaps in 2005 and 2006 cannot be used to test imputation because their true values are unknown. Instead, we select a complete period from 2007 and deliberately hide known observations.

We create:

- one missing hour;
- one six-hour gap;
- one 24-hour gap.

The original observations are stored in `imputation_truth`. The series with hidden values is `corrupted_load`. Pass only `corrupted_load` to the imputation methods.

In [ ]:
imputation_truth = aggregate_load_2007.loc[
    "2007-09-23 00:00":"2007-10-28 23:00"
].copy()

artificial_gaps = {
    "Isolated hour": pd.date_range("2007-10-02 14:00", periods=1, freq="h"),
    "Six-hour gap": pd.date_range("2007-10-10 09:00", periods=6, freq="h"),
    "24-hour gap": pd.date_range("2007-10-20 00:00", periods=24, freq="h"),
}
masked_timestamps = pd.DatetimeIndex(
    sorted(timestamp for gap in artificial_gaps.values() for timestamp in gap)
)

if not masked_timestamps.isin(imputation_truth.index).all():
    raise ValueError("Every masked timestamp must lie inside the complete interval.")
if imputation_truth.loc[masked_timestamps].isna().any():
    raise ValueError("Ground truth must be observed at every masked timestamp.")

corrupted_load = imputation_truth.copy()
corrupted_load.loc[masked_timestamps] = np.nan

print(f"Artificially masked observations: {corrupted_load.isna().sum()}")
for gap_name, gap_index in artificial_gaps.items():
    print(f"  {gap_name}: {gap_index[0]} to {gap_index[-1]} ({len(gap_index)} hour(s))")


### Inspect the artificial gaps

Look at each gap before choosing an imputation method. Consider:

- how quickly demand changes around the gap;
- whether the daily pattern repeats;
- how much information remains when the gap becomes longer.

The shaded areas show the observations that must be reconstructed.

In [ ]:
fig, axes = plt.subplots(
    len(artificial_gaps),
    1,
    figsize=(11, 9),
    sharey=True,
)

for ax, (gap_name, gap_index) in zip(axes, artificial_gaps.items()):
    window_start = gap_index[0] - pd.Timedelta(hours=24)
    window_end = gap_index[-1] + pd.Timedelta(hours=24)
    visible_data = corrupted_load.loc[window_start:window_end]

    ax.plot(
        visible_data.index,
        visible_data,
        color="tab:blue",
        marker="o",
        markersize=2.5,
        linewidth=1.2,
        label="Observed demand",
    )
    ax.axvspan(
        gap_index[0] - pd.Timedelta(hours=1),
        gap_index[-1] + pd.Timedelta(hours=1),
        color="tab:red",
        alpha=0.12,
        label="Artificial gap",
    )
    ax.set_title(gap_name)
    ax.set_ylabel("Demand (MW)")
    ax.grid(alpha=0.2)

axes[0].legend(fontsize=8)
axes[-1].set_xlabel("Timestamp")
fig.suptitle("Corrupted demand series before imputation", fontsize=13)
plt.show()


### Task 1: implement the imputation methods

Implement four methods:

1. forward fill;
2. the mean of the observed values in the selected period;
3. time-based linear interpolation;
4. the observation from the same hour one week earlier, $m=168$.

Each function must return a complete `Series` with the same index. It must not change observed values. Do not use `imputation_truth` inside an imputation function.

In [ ]:
def forward_fill_imputation(series):
    '''Fill each gap using the last observed value.'''
    # TODO: apply forward filling and check that no NaN remains.
    raise NotImplementedError("Implement forward_fill_imputation().")


def observed_mean_imputation(series):
    '''Fill gaps with the mean of all observed values in this interval.'''
    # TODO: calculate the mean from series after ignoring NaN values.
    raise NotImplementedError("Implement observed_mean_imputation().")


def linear_interpolation_imputation(series):
    '''Fill internal gaps by linear interpolation in time.'''
    # TODO: use Series.interpolate with the DatetimeIndex.
    raise NotImplementedError("Implement linear_interpolation_imputation().")


def previous_week_imputation(series, period=168):
    '''Replace each missing value with the observation one week earlier.'''
    # TODO: for each missing timestamp, read from timestamp - period hours.
    # Use only values in series, not imputation_truth.
    raise NotImplementedError("Implement previous_week_imputation().")


In [ ]:
imputed_series = {
    "Forward fill": forward_fill_imputation(corrupted_load),
    "Observed mean": observed_mean_imputation(corrupted_load),
    "Linear interpolation": linear_interpolation_imputation(corrupted_load),
    "Previous week": previous_week_imputation(corrupted_load, period=168),
}

observed_mask = corrupted_load.notna()
for method_name, reconstructed in imputed_series.items():
    if not reconstructed.index.equals(corrupted_load.index):
        raise ValueError(f"{method_name} changed the index.")
    if not reconstructed.loc[observed_mask].equals(corrupted_load.loc[observed_mask]):
        raise ValueError(f"{method_name} changed observations that were not masked.")


### Task 2: evaluate reconstruction accuracy

Complete `score_imputations`. Calculate RMSE and MAE only at the timestamps that were hidden artificially. A lower score means that the reconstructed values are closer to the true observations.

In [ ]:
def score_imputations(truth, missing_index, reconstructions):
    '''Return RMSE and MAE at deliberately masked timestamps only.'''
    # TODO: subset truth and each reconstruction with missing_index.
    # Calculate error as actual - reconstructed and return a DataFrame
    # with methods as rows and RMSE (MW), MAE (MW) as columns.
    raise NotImplementedError("Complete score_imputations().")


In [ ]:
imputation_scores = score_imputations(
    imputation_truth,
    masked_timestamps,
    imputed_series,
)
imputation_scores.round(2)


### Visual comparison

Compare the methods for each gap length. Look for missed peaks, flattened profiles and abrupt changes introduced by the reconstruction.

In [ ]:
def plot_imputation_window(gap_name, gap_index, truth, corrupted, reconstructions):
    '''Plot 24 hours of context on either side of one artificial gap.'''
    window_start = gap_index[0] - pd.Timedelta(hours=24)
    window_end = gap_index[-1] + pd.Timedelta(hours=24)

    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(
        truth.loc[window_start:window_end],
        color="black",
        linewidth=2.2,
        label="True demand",
    )
    ax.plot(
        corrupted.loc[window_start:window_end],
        color="0.65",
        marker="o",
        markersize=2.5,
        linewidth=1,
        label="Corrupted series",
    )
    for method_name, reconstructed in reconstructions.items():
        ax.plot(
            reconstructed.loc[window_start:window_end],
            linewidth=1.2,
            alpha=0.9,
            label=method_name,
        )

    ax.axvspan(
        gap_index[0] - pd.Timedelta(hours=1),
        gap_index[-1] + pd.Timedelta(hours=1),
        color="tab:red",
        alpha=0.08,
        )
    ax.set(title=gap_name, xlabel="Timestamp", ylabel="Demand (MW)")
    ax.grid(alpha=0.2)
    ax.legend(ncol=2, fontsize=8)
    plt.show()


for gap_name, gap_index in artificial_gaps.items():
    plot_imputation_window(
        gap_name,
        gap_index,
        imputation_truth,
        corrupted_load,
        imputed_series,
    )


### Optional extension: average the surrounding weeks

Compare previous-week replacement with the mean of the matching hours in the previous and following weeks. This method is suitable only for retrospective cleaning because it uses observations from after the gap.

In [ ]:
def surrounding_weeks_average_imputation(series, period=168):
    '''Average matching hours in the preceding and following weeks.'''
    # OPTIONAL TODO: use timestamp - period and timestamp + period.
    raise NotImplementedError(
        "Optional: implement surrounding_weeks_average_imputation()."
    )


## Continue with forecasting

The data-quality and imputation tutorial ends here. Continue with [Part 2: Introduction to day-ahead load forecasting](tutorial_1_2_forecast_intro.ipynb), which uses the original clean demand series.